# Day 47: Security & Rate Limiting

Protect your deployed LLM API with API keys and rate limits.

In [ ]:
# 1. FastAPI app with API key security
%%writefile secure_api.py
from fastapi import FastAPI, Depends, HTTPException, Security
from fastapi.security import APIKeyHeader
from slowapi import Limiter, _rate_limit_exceeded_handler
from slowapi.util import get_remote_address
from slowapi.errors import RateLimitExceeded
import os
from dotenv import load_dotenv

load_dotenv()

app = FastAPI()
limiter = Limiter(key_func=get_remote_address)
app.state.limiter = limiter
app.add_exception_handler(RateLimitExceeded, _rate_limit_exceeded_handler)

# API key setup
API_KEY = os.getenv("API_KEY", "my-secret-key")
API_KEY_NAME = "X-API-Key"
api_key_header = APIKeyHeader(name=API_KEY_NAME, auto_error=False)

async def validate_api_key(api_key: str = Security(api_key_header)):
    if api_key is None or api_key != API_KEY:
        raise HTTPException(status_code=403, detail="Invalid API Key")
    return api_key

@app.get("/generate")
@limiter.limit("5/minute")
async def generate(prompt: str, api_key: str = Depends(validate_api_key), request=None):
    # Your LLM call here (mock)
    return {"response": f"You asked: {prompt}", "api_key_used": api_key[:4] + "..."}

if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=8000)
print("Secure API written to secure_api.py")

In [ ]:
# 2. Test the API (run server first: python secure_api.py)
import requests

url = "http://localhost:8000/generate"
headers = {"X-API-Key": "my-secret-key"}
params = {"prompt": "Hello, world!"}

response = requests.get(url, headers=headers, params=params)
print(response.json())

In [ ]:
# 3. Rate limiting test (rapid requests)
for i in range(7):
    resp = requests.get(url, headers=headers, params={"prompt": f"Test {i}"})
    print(f"Request {i}: {resp.status_code}")
    if resp.status_code == 429:
        print("Rate limit hit!")
        break

In [ ]:
# 4. Gradio alternative: add API key and rate limits via queue
# In your Gradio app, you can set:
# demo.queue(default_concurrency_limit=5)
# And use gr.LoginButton for simple authentication (not secure for API).
print("For Gradio Spaces, use environment variables and check inside function.")